In [0]:
import urllib.request

# ── 1. Create the volume ────────────────────────────────────────────────────
spark.sql("CREATE VOLUME IF NOT EXISTS sales.bronze.mappings")
print("Volume sales.bronze.mappings is ready.")

# ── 2. Download Excel file from GitHub into the volume ──────────────────────
GITHUB_URL  = (
    "https://raw.githubusercontent.com/marvinjayson/DATABRICKS-END-TO-END"
    "/main/mapping/master_mapping_bronze_silver_gold_pii.xlsx"
)
DEST_PATH   = "/Volumes/sales/bronze/mappings/master_mapping_bronze_silver_gold_pii.xlsx"

urllib.request.urlretrieve(GITHUB_URL, DEST_PATH)
print(f"File saved to: {DEST_PATH}")

# ── 3. Verify ────────────────────────────────────────────────────────────────
import os
size_kb = os.path.getsize(DEST_PATH) / 1024
print(f"File size: {size_kb:.1f} KB")

In [0]:
%pip install openpyxl pyyaml -q

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

FILE_PATH = "/Volumes/sales/bronze/mappings/master_mapping_bronze_silver_gold_pii.xlsx"

# List all sheets
xl = pd.ExcelFile(FILE_PATH)
print("Sheets:", xl.sheet_names)

# Preview each sheet
for sheet in xl.sheet_names:
    df = xl.parse(sheet)
    print(f"\n--- Sheet: '{sheet}' ({df.shape[0]} rows x {df.shape[1]} cols) ---")
    print(df.head(3).to_string())

In [0]:
import pandas as pd
import yaml
from collections import defaultdict

FILE_PATH   = "/Volumes/sales/bronze/mappings/master_mapping_bronze_silver_gold_pii.xlsx"
YAML_OUTPUT = "/Volumes/sales/bronze/mappings/pipeline_config.yaml"

def clean_df(sheet_name):
    """Read a sheet (header is row index 3 — rows 0-2 are title/subtitle/blank) and drop all-NaN rows."""
    df = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=3)
    return df.dropna(how="all").reset_index(drop=True)

def strip_nan(d: dict) -> dict:
    """Remove keys whose value is NaN / empty string."""
    return {
        k: str(v).strip()
        for k, v in d.items()
        if pd.notna(v) and str(v).strip() not in ("", "nan", "None")
    }

# ── Bronze → Silver ──────────────────────────────────────────────────────────
b2s = clean_df("Bronze_to_Silver")
b2s_map = defaultdict(lambda: {"source_table": None, "columns": []})
for _, row in b2s.iterrows():
    target = row.get("Target Table")
    if pd.isna(target):
        continue
    b2s_map[target]["source_table"] = row.get("Source Table")
    b2s_map[target]["columns"].append(strip_nan({
        "source_column":      row.get("Source Column"),
        "target_column":      row.get("Target Column"),
        "target_data_type":   row.get("Target Data Type"),
        "transformation":     row.get("Transformation Logic"),
        "dq_rule":            row.get("DQ / Filter Rule"),
        "business_definition":row.get("Business Definition"),
        "status":             row.get("Status"),
        "pii_tag":            row.get("PII Tag"),
    }))

# ── Silver → Gold ────────────────────────────────────────────────────────────
s2g = clean_df("Silver_to_Gold")
s2g_map = defaultdict(lambda: {"source_tables": None, "columns": []})
for _, row in s2g.iterrows():
    target = row.get("Target Gold Table")
    if pd.isna(target):
        continue
    s2g_map[target]["source_tables"] = row.get("Source Table(s)")
    s2g_map[target]["columns"].append(strip_nan({
        "source_columns":  row.get("Source Column(s)"),
        "target_column":   row.get("Target Column"),
        "target_data_type":row.get("Target Data Type"),
        "transformation":  row.get("Aggregation / Transformation Logic"),
        "grain":           row.get("Grain"),
        "business_use":    row.get("Business Use"),
        "status":          row.get("Status"),
        "pii_tag":         row.get("PII Tag"),
    }))

# ── DQ Rules ─────────────────────────────────────────────────────────────────
dq = clean_df("DQ_Rules")
dq_map = defaultdict(list)
for _, row in dq.iterrows():
    if pd.isna(row.get("Rule ID")):
        continue
    table = str(row.get("Table", "unknown")).strip()
    dq_map[table].append(strip_nan({
        "rule_id":    row.get("Rule ID"),
        "layer":      row.get("Layer"),
        "columns":    row.get("Column(s)"),
        "rule_type":  row.get("Rule Type"),
        "expectation":row.get("Expectation / Rule"),
        "action":     row.get("Action"),
        "severity":   row.get("Severity"),
        "status":     row.get("Status"),
        "pii_tag":    row.get("PII Tag"),
    }))

# ── Assemble & write YAML ────────────────────────────────────────────────────
config = {
    "pipeline": {
        "name": "olist_lakehouse",
        "description": "Olist e-commerce medallion pipeline — Bronze to Silver to Gold",
        "bronze_to_silver": {
            t: {"source_table": str(m["source_table"]), "columns": m["columns"]}
            for t, m in b2s_map.items()
        },
        "silver_to_gold": {
            t: {"source_tables": str(m["source_tables"]), "columns": m["columns"]}
            for t, m in s2g_map.items()
        },
        "dq_rules": dict(dq_map),
    }
}

with open(YAML_OUTPUT, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

import os
size_kb = os.path.getsize(YAML_OUTPUT) / 1024
with open(YAML_OUTPUT) as f:
    lines = f.readlines()

print(f"YAML written to : {YAML_OUTPUT}")
print(f"File size       : {size_kb:.1f} KB  ({len(lines)} lines)")
print(f"Bronze->Silver  : {len(b2s_map)} target tables")
print(f"Silver->Gold    : {len(s2g_map)} target tables")
print(f"DQ rule groups  : {len(dq_map)} tables\n")
print("--- Preview (first 60 lines) ---")
print("".join(lines[:60]))